<a href="https://colab.research.google.com/github/Lobnaait/SEARCH_Lobna_Tsetline_CMRI/blob/main/Tsetline_ACDCdataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from huggingface_hub import snapshot_download
data_dir = snapshot_download(repo_id="mathpluscode/ACDC", allow_patterns=["*.nii.gz", "*.csv"], repo_type="dataset")
# https://huggingface.co/datasets/mathpluscode/ACDC

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 752 files:   0%|          | 0/752 [00:00<?, ?it/s]

In [2]:
!ls /root/.cache/huggingface/hub/datasets--mathpluscode--ACDC/snapshots/4a312b610f395eda5dc0aa36f087dd6ca9991e41/

test  test.csv	train  train.csv


In [3]:
!pip install pyTsetlinMachine

  Preparing metadata (setup.py) ... done
  Created wheel for pyTsetlinMachine: filename=pytsetlinmachine-0.6.6-cp313-cp313-linux_x86_64.whl size=59790 sha256=1b1a54870c20a2f4f794fe7d31111a3de5dab8a5bf36c06d815c649cb8ccc001
  Stored in directory: /root/.cache/pip/wheels/a9/66/98/535c2cc844fdb6fc12f31feefbaa6a33222b1378914098f498
Successfully built pyTsetlinMachine


In [4]:
#IMPORT LIBRARIES AND DATASET
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, auc, roc_auc_score
from pyTsetlinMachine.tm import MultiClassTsetlinMachine

In [9]:
import os

train = pd.read_csv(os.path.join(data_dir, "train.csv"))
test = pd.read_csv(os.path.join(data_dir, "test.csv"))

print(train.shape)
print(test.shape)

train.head(10)

(100, 15)
(50, 15)


,pid,pathology,height,weight,bmi,n_frames,ed_frame,es_frame,original_sax_spacing_x,original_sax_spacing_y,original_sax_spacing_z,n_slices,edv,esv,ef
0,patient001,DCM,1.84,95.0,28.060019,30,1,12,1.562500,1.562500,10.0,10,1354.58,1125.97,16.876818
1,patient002,DCM,1.60,70.0,27.343750,30,1,12,1.367188,1.367188,10.0,10,1212.69,979.53,19.226678
2,patient003,DCM,1.65,77.0,28.282828,30,1,15,1.562500,1.562500,10.0,10,1405.42,1298.96,7.574960
3,patient004,DCM,1.59,46.0,18.195483,28,1,15,1.367188,1.367188,10.0,10,1226.58,1114.83,9.110698
4,patient005,DCM,1.65,77.0,28.282828,30,1,13,1.406250,1.406250,10.0,10,1446.55,1211.36,16.258684
5,patient006,DCM,1.80,70.0,21.604938,28,1,16,1.757812,1.757812,10.0,11,1637.74,1374.18,16.092909
6,patient007,DCM,1.73,107.0,35.751278,16,1,7,1.875000,1.875000,10.0,10,1589.67,1411.75,11.192260
7,patient008,DCM,1.80,100.0,30.864198,28,1,13,1.562500,1.562500,10.0,10,1332.23,1158.90,13.010516
8,patient009,DCM,1.53,61.0,26.058354,35,1,13,1.367190,1.367190,10.0,10,1254.25,1154.37,7.963325
9,patient010,DCM,1.70,68.0,23.529412,28,1,13,1.562500,1.562500,10.0,10,1484.35,1316.59,11.301917


In [ ]:
# Target/output = second column
y = train.iloc[:, 1]

# Inputs/features = all columns except the second
X = train.drop(train.columns[1], axis=1)

X_train, X_validation, y_train, y_validation = train_test_split(X, y, test_size=0.25)


In [ ]:
# CHOOSE THE TRESHOLDS: Based on the number of treshold divide the dataset into tresholds
def compute_thresholds(X, n_thresholds):
  # Extract non-zero pixel
  non_zero_pixels = X[X > 0]
  percentile_values = np.linspace(0, 100, int(n_thresholds) + 2)[1:-1] # Exclude 0 and 100
  thresholds = np.percentile(non_zero_pixels, percentile_values)
  return np.unique(thresholds)

In [ ]:
def booleanise(X, thresholds):
  # Flatten each image
  X_flat = X.reshape(X.shape[0], -1)
  X_bool = (X_flat[:, :, None] > thresholds[None, None, :]) #Compare every pixel with every threshold
  #convert to binary
  X_bool = X_bool.astype(np.uint32)
  X_bool = X_bool.reshape(X_bool.shape[0], -1)
  return X_bool

In [ ]:
# VALIDATION: Test how many tresholds are optimal to use?
threshold_candidates = np.linspace(1,10,10)
results = []

for n_thresholds in threshold_candidates:

    thresholds = compute_thresholds(X_train, n_thresholds)
    X_train_bool = booleanise(X_train, thresholds)
    X_val_bool = booleanise(X_val, thresholds)

    # model
    tm = MultiClassTsetlinMachine(number_of_clauses=1000, T=50, s=5.0)
    tm.fit(X_train_bool, y_train, epochs=100)

    val_accuracy = accuracy_score(y_val, tm.predict(X_val_bool))

    results.append([n_thresholds, val_accuracy])

In [ ]:
results = np.array(results)
print(results)


[[ 1.          0.94444444]
 [ 2.          0.97037037]
 [ 3.          0.95185185]
 [ 4.          0.96666667]
 [ 5.          0.97037037]
 [ 6.          0.96666667]
 [ 7.          0.97037037]
 [ 8.          0.95925926]
 [ 9.          0.97407407]
 [10.          0.97407407]]


In [ ]:
index = np.argmax(results[:,1]) #index of max accuracy
row_with_max_accuracy = results[index]
print(f"Row with maximum accuracy score: {row_with_max_accuracy}")

Row with maximum accuracy score: [9.         0.97407407]


In [ ]:
# Best treshold
results[index,0]
threshold = compute_thresholds(X_train_full, results[index,0])

In [ ]:
#we retrain the model using all the available data so that the model is trained using all the available data

X_train_full_bool = booleanise(X_train_full, threshold)
X_test_bool = booleanise(X_test, threshold)
tm = MultiClassTsetlinMachine(number_of_clauses=1000, T=50, s=5.0)
tm.fit(X_train_full_bool, y_train_full, epochs=100)
# test the model
predictions = tm.predict(X_test_bool)
print("Accuracy:", accuracy_score(y_test, predictions))


Accuracy: 0.9755555555555555
